# Recurrent Neural Networks with Keras 

By the end, you should be able to:

1. explain the three dimensions of an RNN tensor;
2. distinguish `SimpleRNN`, `GRU`, and `LSTM` at a high level;
3. predict the effect of `return_sequences=True`;
4. explain what an LSTM state contains.


## 1. Setup

The random seeds make repeated runs more consistent. They are useful for teaching comparisons, but they are not required for the code to run.


In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2" #tells TensorFlow to hide a lot of low-level info and warning messages.

import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


SEED = 42
random.seed(SEED) #fixes Python’s built-in random behavior.
np.random.seed(SEED) #fixes NumPy random behavior.
tf.random.set_seed(SEED) #fixes TensorFlow random behavior as much as possible.

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.21.0


## 2. One sequence, one batch

For an MNIST image treated as a sequence:

- **28 timesteps** = 28 image rows;
- **28 features per timestep** = 28 pixel values in each row.

One sample therefore has shape `(28, 28)`. A batch adds a leading dimension: `(batch_size, 28, 28)`.

**Predict before running:** If the batch contains 4 images, what is its shape?


In [2]:
toy_batch = np.zeros((4, 28, 28), dtype=np.float32)

print("Single sample shape:", toy_batch[0].shape)
print("Batch shape:", toy_batch.shape)
print("Meaning: (batch, timesteps, features)")


Single sample shape: (28, 28)
Batch shape: (4, 28, 28)
Meaning: (batch, timesteps, features)


## 3. Three common Keras RNN layers

- `SimpleRNN`: the simplest recurrent layer; useful for learning the basic idea.
- `GRU`: uses gates to control memory; usually fewer parameters than an LSTM with the same number of units.
- `LSTM`: uses gates plus two states; designed to retain useful information over longer sequences.

All three receive a 3D tensor: `(batch, timesteps, features)`.

The following models use only 8 recurrent units so their summaries are easy to inspect.


In [3]:
def make_classifier(recurrent_layer, model_name):
    return keras.Sequential([
        layers.Input(shape=(28, 28), name="image_sequence"),
        recurrent_layer,
        layers.Dense(10, activation="softmax", name="class_probabilities"),
    ], name=model_name)


simple_model = make_classifier(layers.SimpleRNN(8, name="simple_rnn"), "simple_rnn_model")
gru_model = make_classifier(layers.GRU(8, name="gru"), "gru_model")
lstm_model = make_classifier(layers.LSTM(8, name="lstm"), "lstm_model")

for current_model in [simple_model, gru_model, lstm_model]:
    print("\n", current_model.name)
    current_model.summary()



 simple_rnn_model


Model: "simple_rnn_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)               │ (None, 8)                   │             296 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ class_probabilities (Dense)          │ (None, 10)                  │              90 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 386 (1.51 KB)

 Trainable params: 386 (1.51 KB)

...

Model: "lstm_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 8)                   │           1,184 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ class_probabilities (Dense)          │ (None, 10)                  │              90 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,274 (4.98 KB)

 Trainable params: 1,274 (4.98 KB)

 Non-trainable params: 0 (0.00 B)

### Parameter-count checkpoint

Let `F` be the number of input features and `U` the number of recurrent units.

- SimpleRNN: `U × (F + U + 1)`
- GRU in Keras (default `reset_after=True`): `3U(F + U) + 6U`
- LSTM: `4U(F + U + 1)`

For the LSTM above, `F=28` and `U=8`, so the LSTM has `4 × 8 × (28 + 8 + 1) = 1,184` parameters.

**Think:** Why does the LSTM multiply by 4? Because it calculates four groups: the input gate, forget gate, output gate, and candidate memory.


## 4. `return_sequences`: last output or every output?

The default is `return_sequences=False`:

- output shape: `(batch, units)`;
- only the output at the last timestep is returned.

With `return_sequences=True`:

- output shape: `(batch, timesteps, units)`;
- one output vector is returned for every timestep.

**Predict before running:** For batch size 4, 28 timesteps and 8 units, predict both shapes and their ranks.


In [4]:
last_output_layer = layers.LSTM(8, return_sequences=False, name="last_output")
all_outputs_layer = layers.LSTM(8, return_sequences=True, name="all_outputs")

last_output = last_output_layer(toy_batch)
all_outputs = all_outputs_layer(toy_batch)

print("return_sequences=False:", last_output.shape, "rank =", last_output.ndim)
print("return_sequences=True: ", all_outputs.shape, "rank =", all_outputs.ndim)


return_sequences=False: (4, 8) rank = 2
return_sequences=True:  (4, 28, 8) rank = 3


### Why stacked RNNs need `return_sequences=True`

If another recurrent layer comes next, it needs a sequence, not just one final vector. Therefore, the first recurrent layer must return all timestep outputs.

`(batch, 28, 16) → (batch, 8) → (batch, 10)`

The first LSTM keeps the 28 timesteps; the second LSTM reduces them to one vector.


In [5]:
stacked_model = keras.Sequential([
    layers.Input(shape=(28, 28), name="image_sequence"),
    layers.LSTM(16, return_sequences=True, name="lstm_sequence"),
    layers.LSTM(8, name="lstm_summary"),
    layers.Dense(10, activation="softmax", name="class_probabilities"),
], name="stacked_lstm_demo")

stacked_model.summary()


Model: "stacked_lstm_demo"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_sequence (LSTM)                 │ (None, 28, 16)              │           2,880 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_summary (LSTM)                  │ (None, 8)                   │             800 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ class_probabilities (Dense)          │ (None, 10)                  │              90 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,770 (14.73 KB)

 Trainable params: 3,770 (14.73 KB)

 Non-trainable params: 0 (0.00 B)

## 5. `return_state`: output, hidden state and cell state

An LSTM can return three tensors:

```text
output, state_h, state_c
```

- `output`: the layer output at the last timestep;
- `state_h`: final hidden state (short-term working representation);
- `state_c`: final cell state (longer-term memory path).

For a single LSTM layer with `return_sequences=False`, `output` and `state_h` contain the same final hidden values. They are returned separately because the state can be passed to another RNN or used to resume processing.


In [6]:
state_demo = layers.LSTM(8, return_state=True, name="state_demo")
output, state_h, state_c = state_demo(toy_batch)

print("output shape: ", output.shape)
print("state_h shape:", state_h.shape)
print("state_c shape:", state_c.shape)
print("output equals state_h:", np.allclose(output.numpy(), state_h.numpy()))


output shape:  (4, 8)
state_h shape: (4, 8)
state_c shape: (4, 8)
output equals state_h: True


In [7]:
bidirectional_model = keras.Sequential([
    layers.Input(shape=(28, 28), name="image_sequence"),
    layers.Bidirectional(layers.LSTM(8), name="bidirectional_lstm"),
    layers.Dense(10, activation="softmax", name="class_probabilities"),
], name="bidirectional_demo")

bidirectional_model.summary()


Model: "bidirectional_demo"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ bidirectional_lstm (Bidirectional)   │ (None, 16)                  │           2,368 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ class_probabilities (Dense)          │ (None, 10)                  │             170 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,538 (9.91 KB)

 Trainable params: 2,538 (9.91 KB)

 Non-trainable params: 0 (0.00 B)

### When bidirectional processing is appropriate

It is useful when the whole sequence is already available, such as classifying a complete sentence or image.

It is not appropriate when a prediction must use only the past. For example, a real-time next-step forecast must not look at future timesteps.


## 6. Questions:

Write short answers for the questions below: 

1. What does each dimension in `(None, 28, 64)` mean?
2. What changes when `return_sequences=True`?
3. Why does a stacked RNN usually need `return_sequences=True` in its first recurrent layer?
4. State one way evaluation information could leak into training.
